# Lofty ACE-Step Worker — Google Colab

T4 GPU → удалённый воркер для генерации музыки и файнтюнинга ACE-Step 1.5.

**Оптимизации:**
- CPU offload (LM/VAE выгружаются в RAM — экономия ~4 GB VRAM)
- torch.compile для DiT (~20-30% ускорение после прогрева)
- Flash/SDPA Attention (~15-20%)
- Файнтюнинг LoRA/LoKR через нативный ACE-Step training API
- NaN retry с fallback на инструментал

---

## Подготовка (на вашем ПК)

1. `docker compose up -d`
2. `ngrok http 8000`
3. Скопируйте ngrok URL ниже

**Запускайте по порядку: Шаг 1 → 2 → 3 → 4 → 5**

## Шаг 1. Конфигурация

In [ ]:
import torch, requests, os

# ╔════════════════════════════════════════════════╗
# ║  CONFIGURE                                    ║
# ╚════════════════════════════════════════════════╝
API_URL = "https://YOUR-NGROK-URL.ngrok-free.dev"
WORKER_API_KEY = "lofty-colab-worker-2026"
POLL_INTERVAL = 3
# ════════════════════════════════════════════════

assert torch.cuda.is_available(), "GPU не найден! Runtime -> Change runtime type -> T4 GPU"
gpu = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU: {gpu} ({vram:.1f} GB) | CUDA {torch.version.cuda}")

try:
    r = requests.get(f"{API_URL}/health", timeout=10,
        headers={"ngrok-skip-browser-warning": "true", "User-Agent": "LoftyWorker/2.0"})
    print(f"API: {'OK' if r.status_code == 200 else r.status_code}")
except Exception as e:
    print(f"API недоступен: {e}\nПроверьте: docker compose up -d + ngrok http 8000")

## Шаг 2. Установка ACE-Step 1.5

In [ ]:
%%bash
set -e
echo "=== ACE-Step 1.5 ==="

[ ! -d /content/ACE-Step-1.5 ] && \
    git clone --depth 1 https://github.com/ace-step/ACE-Step-1.5.git /content/ACE-Step-1.5 || \
    echo "Already cloned"

cd /content/ACE-Step-1.5
pip install -q -e ".[all]" 2>/dev/null || pip install -q -e . --no-deps

pip install -q \
    "transformers>=4.51.0,<4.58.0" "tokenizers>=0.22.0,<=0.23.0" \
    accelerate diffusers einops loguru peft scipy soundfile \
    vector_quantize_pytorch diskcache lycoris-lora lightning \
    huggingface_hub 2>/dev/null

for d in acestep/third_parts/nano-vllm third_parts/nano-vllm; do
    [ -d "$d" ] && cd "$d" && pip install -q . 2>/dev/null && break
done

which ffmpeg >/dev/null 2>&1 || apt-get -qq install -y ffmpeg >/dev/null 2>&1
echo "=== Done ==="

## Шаг 3. Патчи и модели

In [ ]:
import os, sys, shutil, subprocess, importlib, re

ACESTEP_SRC = "/content/ACE-Step-1.5/acestep"
SITE_PKG = "/usr/local/lib/python3.12/dist-packages/acestep"
CHECKPOINT_DIR = "/usr/local/lib/python3.12/dist-packages/checkpoints"

# 1. debug_utils stubs
print("[1/4] debug_utils...")
all_grep = ""
for d in [ACESTEP_SRC, ACESTEP_SRC + "/../acestep/third_parts/"]:
    if os.path.isdir(d):
        r = subprocess.run(["grep", "-roh", "from acestep.debug_utils import.*", d],
                          capture_output=True, text=True)
        all_grep += r.stdout + "\n"

names = set()
for line in all_grep.split("\n"):
    for m in re.findall(r'import\s+([a-z_,\s]+)', line):
        for n in m.split(","):
            n = n.strip()
            if n and n.isidentifier(): names.add(n)

with open(os.path.join(ACESTEP_SRC, "debug_utils.py"), "w") as f:
    f.write("# Auto-generated stubs\n\n")
    for n in sorted(names): f.write(f"def {n}(*a, **kw): pass\n\n")
    f.write("def __getattr__(name):\n    def _s(*a, **kw): pass\n    return _s\n")
print(f"  {len(names)} stubs")

# 2. site-packages symlink
print("[2/4] site-packages...")
if os.path.isdir(SITE_PKG) and not os.path.islink(SITE_PKG): shutil.rmtree(SITE_PKG)
if not os.path.exists(SITE_PKG): os.symlink(ACESTEP_SRC, SITE_PKG)
print("  OK")

# 3. nano-vllm
print("[3/4] nano-vllm...")
nano = "/content/ACE-Step-1.5/acestep/third_parts/nano-vllm"
if os.path.isdir(nano):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", nano], capture_output=True)
importlib.invalidate_caches()
for k in [k for k in list(sys.modules) if "acestep" in k or "nanovllm" in k]: del sys.modules[k]
try:
    import nanovllm; print("  nanovllm: OK")
except ImportError as e:
    print(f"  nanovllm: {e} (PyTorch fallback)")

# 4. Models
print("[4/4] Models...")
models_ok = True
for sub, min_gb in [("acestep-v15-turbo", 4.0), ("acestep-5Hz-lm-1.7B", 3.0), ("vae", 0.2)]:
    p = os.path.join(CHECKPOINT_DIR, sub)
    if os.path.isdir(p):
        sz = sum(os.path.getsize(os.path.join(p, f)) for f in os.listdir(p)
                 if os.path.isfile(os.path.join(p, f))) / 1024**3
        ok = sz >= min_gb and any("safetensors" in f or "pytorch_model" in f for f in os.listdir(p))
        print(f"  {sub}: {sz:.2f} GB [{'OK' if ok else 'INCOMPLETE'}]")
        if not ok: models_ok = False
    else:
        print(f"  {sub}: MISSING"); models_ok = False

if not models_ok:
    print("\n  Downloading models (10-15 min)...")
    from huggingface_hub import snapshot_download
    snapshot_download(repo_id="ACE-Step/Ace-Step1.5", local_dir=CHECKPOINT_DIR, local_dir_use_symlinks=False)
    lm = os.path.join(CHECKPOINT_DIR, "acestep-5Hz-lm-1.7B")
    if not (os.path.isdir(lm) and any("safetensors" in f for f in os.listdir(lm))):
        snapshot_download(repo_id="ACE-Step/acestep-5Hz-lm-1.7B", local_dir=lm, local_dir_use_symlinks=False)
    print("  Done!")
else:
    print("  All models present")

print("\n-> Step 4")

## Шаг 4. Запуск ACE-Step (оптимизированный)

CPU offload + torch.compile + Flash Attention

In [ ]:
import os, sys, subprocess, time, requests, torch, json

PROJECT_ROOT = "/content/ACE-Step-1.5"
CHECKPOINT_DIR = "/usr/local/lib/python3.12/dist-packages/checkpoints"
LOG_FILE = "/content/ace_step_server.log"
ACE_API = "http://localhost:8001"

os.environ["ACE_STEP_PROJECT_ROOT"] = PROJECT_ROOT
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:128"
os.environ.pop("ACESTEP_INIT_LLM", None)
sys.path.insert(0, PROJECT_ROOT)

# Kill old
subprocess.run(["pkill", "-f", "acestep|ace_step"], capture_output=True)
torch.cuda.empty_cache(); time.sleep(2)

vram = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1024**3
print(f"VRAM: {vram:.1f} GB")

# env_bool helper: reads env var as bool, used by run_api_server_main
ENV_BOOL_HELPER = '''
def env_bool(key, default=False):
    val = os.environ.get(key, "")
    if not val:
        return default
    return val.lower() in ("1", "true", "yes", "on")
'''

# Startup script
STARTUP = f'''
import sys, os
sys.path.insert(0, "{PROJECT_ROOT}")
os.environ["ACE_STEP_PROJECT_ROOT"] = "{PROJECT_ROOT}"
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True,max_split_size_mb:128")

def env_bool(key, default=False):
    val = os.environ.get(key, "")
    if not val:
        return default
    return val.lower() in ("1", "true", "yes", "on")

from acestep.api_server import run_api_server_main

try:
    run_api_server_main(env_bool)
except TypeError:
    run_api_server_main()
'''

startup_path = "/content/ace_step_startup.py"
with open(startup_path, "w") as f: f.write(STARTUP)

# Fallback one-liner
FALLBACK_CMD = (
    "import os; "
    "env_bool = lambda key, default=False: os.environ.get(key, '').lower() in ('1','true','yes','on') if os.environ.get(key) else default; "
    "from acestep.api_server import run_api_server_main; "
    "run_api_server_main(env_bool)"
)

def start_ace_server(log_mode="w"):
    """Start ACE-Step server, returns True on success."""
    cmd = [sys.executable, startup_path, "--port", "8001"]
    print(f"Starting server...")
    proc = subprocess.Popen(cmd, stdout=open(LOG_FILE, log_mode), stderr=subprocess.STDOUT,
                           text=True, cwd=PROJECT_ROOT, env={**os.environ})
    for i in range(300):
        if proc.poll() is not None:
            print(f"Crashed (code {proc.returncode}). Trying fallback...")
            with open(LOG_FILE) as f: print(f.read()[-2000:])
            cmd2 = [sys.executable, "-c", FALLBACK_CMD, "--port", "8001"]
            proc = subprocess.Popen(cmd2, stdout=open(LOG_FILE, "a"), stderr=subprocess.STDOUT,
                                   text=True, cwd=PROJECT_ROOT, env={**os.environ})
            for j in range(150):
                try:
                    if requests.get(f"{ACE_API}/health", timeout=2).status_code == 200:
                        print(f"Ready (fallback, {j*2}s)"); return True
                except: pass
                time.sleep(2)
            return False
        try:
            if requests.get(f"{ACE_API}/health", timeout=2).status_code == 200:
                print(f"Ready! ({i*2}s)"); return True
        except: pass
        if i % 15 == 0 and i > 0:
            try:
                with open(LOG_FILE) as f:
                    lines = f.readlines()
                    if lines: print(f"  [{i*2}s] {lines[-1].strip()[:120]}")
            except: pass
        time.sleep(2)
    proc.terminate()
    return False

if not start_ace_server():
    raise RuntimeError("ACE-Step failed to start in 10 min")

# Show optimizations
with open(LOG_FILE) as f: log = f.read()
opts = []
if "offload" in log.lower(): opts.append("CPU offload")
if "compile" in log.lower(): opts.append("torch.compile")
if "flash" in log.lower() or "sdpa" in log.lower(): opts.append("Flash/SDPA")
print(f"Optimizations: {', '.join(opts) or 'standard'}")

# Test generation
def test_gen(prompt, lyrics="", label="test"):
    t0 = time.time()
    payload = {"prompt": prompt, "lyrics": lyrics, "thinking": False,
               "use_cot_caption": False, "use_cot_language": False,
               "audio_duration": 15, "inference_steps": 8,
               "guidance_scale": 5.0, "audio_format": "mp3", "batch_size": 1}
    if lyrics: payload["vocal_language"] = "en"
    r = requests.post(f"{ACE_API}/release_task", json=payload, timeout=60)
    if r.status_code != 200: print(f"  {label}: release_task {r.status_code}"); return False
    tid = r.json()["data"]["task_id"]
    for i in range(60):
        time.sleep(3)
        r2 = requests.post(f"{ACE_API}/query_result",
                          json={"task_id_list": json.dumps([tid])}, timeout=15)
        data = r2.json().get("data", [])
        if not data: continue
        item = data[0]
        try: rl = json.loads(item.get("result", "[]"))
        except: rl = []
        if isinstance(rl, list) and rl and rl[0].get("file"):
            print(f"  {label}: OK {time.time()-t0:.1f}s"); return True
        if item.get("status") == 2:
            print(f"  {label}: NaN/error"); return False
    print(f"  {label}: timeout"); return False

print("\nTests:")
ok1 = test_gen("upbeat electronic dance music", label="instrumental")
if ok1:
    test_gen("rap hip hop", "[Verse]\nYo check it\n[Chorus]\nFeel the beat", label="lyrics")
    test_gen("calm ambient piano", label="warmed-up")

print("\n-> Step 5")

## Шаг 5. Воркер

Оставьте эту ячейку работающей. Воркер выполняет:
1. **Генерацию** — ACE-Step turbo (8 steps)
2. **Файнтюнинг LoRA/LoKR** — обучение на своих треках

In [ ]:
import json, time, requests, subprocess, os, sys, torch, random, shutil, tempfile
from datetime import datetime
from urllib.parse import quote

PROJECT_ROOT = "/content/ACE-Step-1.5"
CHECKPOINT_DIR = "/usr/local/lib/python3.12/dist-packages/checkpoints"
ACE_API = "http://localhost:8001"
LOG_FILE = "/content/ace_step_server.log"
HEADERS = {"Authorization": f"Bearer {WORKER_API_KEY}",
           "ngrok-skip-browser-warning": "true", "User-Agent": "LoftyWorker/2.0"}

MAX_STEPS = 8
MAX_NAN_RETRIES = 3

# Fallback command with env_bool helper
FALLBACK_CMD = (
    "import os; "
    "os.environ['ACESTEP_INIT_LLM']='0'; "
    "env_bool = lambda key, default=False: os.environ.get(key, '').lower() in ('1','true','yes','on') if os.environ.get(key) else default; "
    "from acestep.api_server import run_api_server_main; "
    "run_api_server_main(env_bool)"
)


# ====== ACE-Step Server ======

def ensure_ace_server():
    try:
        if requests.get(f"{ACE_API}/health", timeout=3).status_code == 200: return True
    except: pass
    print("  [!] Restarting ACE-Step...", flush=True)
    subprocess.run(["pkill", "-f", "acestep|ace_step"], capture_output=True)
    torch.cuda.empty_cache(); time.sleep(2)
    startup = "/content/ace_step_startup.py"
    if os.path.exists(startup):
        cmd = [sys.executable, startup, "--port", "8001"]
    else:
        cmd = [sys.executable, "-c", FALLBACK_CMD, "--port", "8001"]
    env = {**os.environ}; env["ACESTEP_INIT_LLM"] = "0"
    env["ACE_STEP_PROJECT_ROOT"] = PROJECT_ROOT
    subprocess.Popen(cmd, stdout=open(LOG_FILE, "a"), stderr=subprocess.STDOUT,
                     text=True, cwd=PROJECT_ROOT, env=env)
    for i in range(150):
        try:
            if requests.get(f"{ACE_API}/health", timeout=2).status_code == 200:
                print(f"  [!] Restarted ({i*2}s)", flush=True); return True
        except: pass
        time.sleep(2)
    print("  [!] Failed to restart", flush=True); return False


def stop_ace_server():
    print("  [!] Stopping ACE-Step for training...", flush=True)
    subprocess.run(["pkill", "-f", "acestep|ace_step"], capture_output=True)
    torch.cuda.empty_cache(); time.sleep(3)
    vram = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1024**3
    print(f"  [!] VRAM free: {vram:.1f} GB", flush=True)


# ====== Music Generation ======

def send_ace_task(payload):
    resp = requests.post(f"{ACE_API}/release_task", json=payload, timeout=60)
    if resp.status_code != 200:
        raise RuntimeError(f"release_task: {resp.status_code} {resp.text[:200]}")
    tid = resp.json()["data"]["task_id"]
    has_lyr = "lyrics" if payload.get("lyrics", "").strip() else "instrumental"
    print(f"  ACE: {tid[:8]}... ({has_lyr}, seed={payload.get('seed','auto')})", flush=True)

    for i in range(120):
        time.sleep(3)
        try:
            r = requests.post(f"{ACE_API}/query_result",
                             json={"task_id_list": json.dumps([tid])}, timeout=15)
            if r.status_code != 200: continue
            data = r.json().get("data", [])
            if not data:
                if i % 10 == 0: print(f"    {i*3}s...", flush=True)
                continue
            item = data[0]
            try: rl = json.loads(item.get("result", "[]")) if isinstance(item.get("result"), str) else item.get("result", [])
            except: rl = []
            if not isinstance(rl, list) or not rl:
                if i % 10 == 0: print(f"    [{i*3}s] status={item.get('status')}", flush=True)
                continue
            first = rl[0]
            fp = first.get("file", "")
            if item.get("status") == 2:
                raise RuntimeError(f"ACE-Step: {first.get('error') or first.get('stage','fail')}")
            if item.get("status") == 1 and fp:
                url = f"{ACE_API}{fp}" if fp.startswith("/v1/audio") else f"{ACE_API}/v1/audio?path={quote(fp)}"
                ar = requests.get(url, timeout=30)
                if ar.status_code == 200:
                    dur = first.get("metas", {}).get("duration", payload.get("audio_duration", 30))
                    return ar.content, 48000, dur, "mp3"
                raise RuntimeError(f"Audio download: {ar.status_code}")
        except requests.exceptions.ConnectionError:
            raise RuntimeError("ACE-Step server down")
        except requests.exceptions.RequestException: continue
    raise RuntimeError("Generation timeout 6min")


def generate_music(job):
    if not ensure_ace_server(): raise RuntimeError("ACE-Step unavailable")
    params = job.get("generation_params") or {}
    lyrics = job.get("lyrics") or ""
    prompt = str(job.get("prompt") or "")
    steps = min(params.get("inference_steps", 8), MAX_STEPS)

    payload = {"prompt": prompt, "lyrics": lyrics, "thinking": False,
               "use_cot_caption": False, "use_cot_language": False, "sample_mode": False,
               "audio_duration": job.get("duration_seconds", 30),
               "inference_steps": steps, "guidance_scale": params.get("guidance_scale", 5.0),
               "task_type": "text2music", "audio_format": "mp3",
               "vocal_language": params.get("language", "en") or "en", "batch_size": 1}
    for k, pk in [("bpm","bpm"),("key","key_scale"),("time_signature","time_signature")]:
        if params.get(k): payload[pk] = params[k]
    seed = params.get("seed", -1)
    if seed is not None and int(seed) >= 0: payload["seed"] = int(seed)
    has_lyr = bool(lyrics.strip())

    for attempt in range(MAX_NAN_RETRIES):
        try: return send_ace_task(payload)
        except RuntimeError as e:
            if "NaN" not in str(e) and "failed" not in str(e).lower(): raise
            ns = random.randint(0, 999999)
            if attempt == MAX_NAN_RETRIES - 2 and has_lyr:
                print(f"  [!] NaN retry {attempt+1}, dropping lyrics", flush=True); payload["lyrics"] = ""
            elif attempt < MAX_NAN_RETRIES - 1:
                print(f"  [!] NaN retry {attempt+1}, seed={ns}", flush=True)
            else: raise
            payload["seed"] = ns; time.sleep(3)
            if not ensure_ace_server(): raise RuntimeError("ACE-Step restart failed")
    raise RuntimeError(f"NaN after {MAX_NAN_RETRIES} retries")


def handle_generation_job(job):
    jid = job["job_id"]
    print(f"\n[{datetime.now():%H:%M:%S}] Gen: {str(jid)[:8]}...", flush=True)
    print(f"  {job['prompt'][:60]} | {job.get('duration_seconds',30)}s", flush=True)
    try: requests.post(f"{API_URL}/api/v1/worker/{jid}/progress", json={"progress":5}, headers=HEADERS, timeout=5)
    except: pass

    t0 = time.time()
    audio, sr, dur, fmt = generate_music(job)
    gt = time.time() - t0

    try: requests.post(f"{API_URL}/api/v1/worker/{jid}/progress", json={"progress":90}, headers=HEADERS, timeout=5)
    except: pass

    r = requests.post(f"{API_URL}/api/v1/worker/{jid}/result", headers=HEADERS,
                      files={"audio_file": (f"track.{fmt}", audio, "audio/mpeg")},
                      data={"status":"completed","duration":str(dur),"sample_rate":str(sr),"format":fmt}, timeout=60)
    if r.status_code != 200: raise RuntimeError(f"Upload: {r.status_code}")
    return gt


# ====== Fine-Tuning ======

def ft_progress(jid, pct):
    try: requests.post(f"{API_URL}/api/v1/worker/{jid}/finetune-progress",
                       json={"progress":pct}, headers=HEADERS, timeout=5)
    except: pass

def ft_cancelled(jid):
    try:
        r = requests.get(f"{API_URL}/api/v1/worker/{jid}/finetune-cancelled", headers=HEADERS, timeout=5)
        return r.status_code == 200 and r.json().get("cancelled", False)
    except: return False

def download_track(storage_key, dest):
    r = requests.get(f"{API_URL}/api/v1/worker/download/{storage_key}", headers=HEADERS, timeout=120, allow_redirects=True)
    if r.status_code != 200: raise RuntimeError(f"Download {r.status_code}: {storage_key}")
    with open(dest, "wb") as f: f.write(r.content)
    return len(r.content)


def prepare_dataset(tracks, tmp):
    """Download tracks and convert all to .wav for DatasetBuilder compatibility."""
    ddir = os.path.join(tmp, "dataset"); os.makedirs(ddir, exist_ok=True)
    for i, t in enumerate(tracks):
        name = f"track_{i:03d}"
        ext = t.get("format", "wav")
        raw_path = os.path.join(ddir, f"{name}_raw.{ext}")
        wav_path = os.path.join(ddir, f"{name}.wav")
        sz = download_track(t["storage_key"], raw_path)
        print(f"    Track {i+1}/{len(tracks)}: {sz/1024:.0f} KB ({ext})", flush=True)
        # Convert to wav via ffmpeg for reliable format support
        if ext.lower() != "wav":
            ret = subprocess.run(
                ["ffmpeg", "-y", "-i", raw_path, "-ar", "48000", "-ac", "2", wav_path],
                capture_output=True, text=True, timeout=120)
            if ret.returncode == 0:
                os.remove(raw_path)
                print(f"      -> converted to wav", flush=True)
            else:
                # ffmpeg failed, try renaming as-is
                print(f"      -> ffmpeg failed: {ret.stderr[:200]}", flush=True)
                os.rename(raw_path, wav_path)
        else:
            os.rename(raw_path, wav_path)
        # Write lyrics sidecar
        lyr = t.get("lyrics", "")
        if lyr.strip():
            with open(os.path.join(ddir, f"{name}.txt"), "w", encoding="utf-8") as f: f.write(lyr)
    # Debug: list final files
    final_files = os.listdir(ddir)
    print(f"    Dataset dir: {final_files}", flush=True)
    return ddir


def handle_finetune_job(ft):
    jid = str(ft["job_id"])
    tracks = ft["track_data"]
    cfg = ft["config"]
    method = cfg.get("training_method", "lokr")
    epochs = cfg.get("max_epochs", 100)
    bs = cfg.get("batch_size", 1)
    lr = cfg.get("learning_rate", 1e-4)

    print(f"\n[{datetime.now():%H:%M:%S}] FineTune: {jid[:8]}...", flush=True)
    print(f"  {ft['job_name']} | {len(tracks)} tracks | {method} | {epochs} epochs", flush=True)
    ft_progress(jid, 0)
    tmp = tempfile.mkdtemp(prefix="lofty_ft_")

    try:
        print("  Downloading tracks...", flush=True); ft_progress(jid, 5)
        ddir = prepare_dataset(tracks, tmp); ft_progress(jid, 10)
        if ft_cancelled(jid):
            requests.post(f"{API_URL}/api/v1/worker/{jid}/finetune-result",
                         headers=HEADERS, data={"status":"cancelled"}, timeout=10); return 0

        stop_ace_server()

        print("  Loading models...", flush=True); ft_progress(jid, 12)
        from acestep.handler import AceStepHandler
        from acestep.training import (DatasetBuilder, AudioSample, LoRATrainer, LoKRTrainer,
                                      LoRAConfig, LoKRConfig, TrainingConfig)

        handler = AceStepHandler()
        msg, ok = handler.initialize_service(project_root=PROJECT_ROOT, config_path="acestep-v15-turbo",
                                             device="cuda", offload_to_cpu=True)
        print(f"    {msg[:80]}", flush=True)
        if not ok: raise RuntimeError(f"Model init failed: {msg}")
        ft_progress(jid, 20)

        print("  Building dataset...", flush=True)
        builder = DatasetBuilder()
        samples, scan_msg = builder.scan_directory(ddir)
        print(f"    Found {len(samples)} audio files ({scan_msg})", flush=True)
        if not samples: raise RuntimeError("No audio files found")

        for i, t in enumerate(tracks):
            if i >= len(samples): break
            kw = {"labeled": True}
            if t.get("caption"): kw["caption"] = t["caption"]
            lyr = t.get("lyrics", "")
            if lyr.strip(): kw["lyrics"] = lyr; kw["is_instrumental"] = False
            else: kw["is_instrumental"] = True
            if t.get("bpm"): kw["bpm"] = int(t["bpm"])
            if t.get("key_scale"): kw["keyscale"] = t["key_scale"]
            builder.update_sample(i, **kw)
        for i in range(len(samples)):
            if not builder._samples[i].labeled: builder.update_sample(i, labeled=True)
        ft_progress(jid, 25)

        if ft_cancelled(jid):
            requests.post(f"{API_URL}/api/v1/worker/{jid}/finetune-result",
                         headers=HEADERS, data={"status":"cancelled"}, timeout=10); return 0

        print("  Preprocessing to tensors...", flush=True)
        tdir = os.path.join(tmp, "tensors"); os.makedirs(tdir, exist_ok=True)
        pmode = "lokr" if method == "lokr" else "lora"
        paths, pmsg = builder.preprocess_to_tensors(
            dit_handler=handler, output_dir=tdir, max_duration=240.0,
            preprocess_mode=pmode, progress_callback=lambda m: print(f"    {m}", flush=True))
        print(f"    {len(paths)} tensors ready", flush=True)
        if not paths: raise RuntimeError("Preprocessing produced no tensors")
        ft_progress(jid, 40)
        del builder; torch.cuda.empty_cache()

        if ft_cancelled(jid):
            requests.post(f"{API_URL}/api/v1/worker/{jid}/finetune-result",
                         headers=HEADERS, data={"status":"cancelled"}, timeout=10); return 0

        print(f"  Training ({method})...", flush=True)
        odir = os.path.join(tmp, "output"); os.makedirs(odir, exist_ok=True)
        tcfg = TrainingConfig(
            learning_rate=lr, batch_size=bs, max_epochs=epochs, output_dir=odir,
            gradient_accumulation_steps=4, save_every_n_epochs=max(epochs//5, 10),
            warmup_steps=min(50, epochs * max(len(paths),1) // 2),
            mixed_precision="bf16", gradient_checkpointing=True, num_workers=2, seed=42)

        t0 = time.time()
        if method == "lokr":
            trainer = LoKRTrainer(handler, LoKRConfig(linear_dim=64, linear_alpha=128), tcfg)
        else:
            trainer = LoRATrainer(handler, LoRAConfig(r=8, alpha=16, dropout=0.1), tcfg)

        state = {"stop": False}
        last_pct = 40
        total_est = epochs * max(len(paths), 1)

        for step, loss, smsg in trainer.train_from_preprocessed(tensor_dir=tdir, training_state=state):
            pct = min(40 + int(min(step / max(total_est, 1), 1.0) * 50), 90)
            if pct - last_pct >= 3:
                ft_progress(jid, pct); last_pct = pct
                print(f"    Step {step}: loss={loss:.4f} ({pct}%)", flush=True)
            if step % 20 == 0 and ft_cancelled(jid):
                state["stop"] = True; trainer.stop()
                requests.post(f"{API_URL}/api/v1/worker/{jid}/finetune-result",
                             headers=HEADERS, data={"status":"cancelled"}, timeout=10); return 0

        tt = time.time() - t0
        print(f"  Training done in {tt:.0f}s", flush=True); ft_progress(jid, 90)
        del trainer; torch.cuda.empty_cache()

        print("  Uploading adapter...", flush=True)
        af = None
        for ext in [".safetensors", ".bin", ".pt"]:
            for root, _, files in os.walk(odir):
                for f in files:
                    if f.endswith(ext): af = os.path.join(root, f); break
                if af: break
            if af: break
        if not af:
            all_f = [os.path.join(r,f) for r,_,fs in os.walk(odir) for f in fs]
            print(f"    Files: {all_f[:10]}", flush=True)
            raise RuntimeError("No adapter file produced")

        asz = os.path.getsize(af)
        print(f"    {os.path.basename(af)} ({asz/1024:.0f} KB)", flush=True)
        with open(af, "rb") as f: abytes = f.read()

        r = requests.post(f"{API_URL}/api/v1/worker/{jid}/finetune-result", headers=HEADERS,
                          files={"adapter_file": ("adapter_model.safetensors", abytes, "application/octet-stream")},
                          data={"status":"completed","training_method":method,"num_tracks":str(len(tracks))}, timeout=120)
        if r.status_code != 200: raise RuntimeError(f"Upload: {r.status_code} {r.text[:200]}")

        ft_progress(jid, 100)
        del handler; torch.cuda.empty_cache()
        return tt

    except KeyboardInterrupt:
        requests.post(f"{API_URL}/api/v1/worker/{jid}/finetune-result",
                     headers=HEADERS, data={"status":"cancelled"}, timeout=10); return 0
    except Exception as e:
        print(f"  Error: {e}", flush=True)
        import traceback; traceback.print_exc()
        try: requests.post(f"{API_URL}/api/v1/worker/{jid}/finetune-result", headers=HEADERS,
                          data={"status":"failed","error_message":str(e)[:500]}, timeout=10)
        except: pass
        raise
    finally:
        if os.path.exists(tmp): shutil.rmtree(tmp, ignore_errors=True)
        torch.cuda.empty_cache()
        print("  Restarting ACE-Step...", flush=True); ensure_ace_server()


# ====== Main Loop ======
print("=" * 50, flush=True)
print(f"  Lofty ACE-Step Worker", flush=True)
print(f"  API: {API_URL}", flush=True)
print(f"  Poll: {POLL_INTERVAL}s | NaN retry: {MAX_NAN_RETRIES}", flush=True)
print(f"  Fine-tuning: LoRA/LoKR via ACE-Step training API", flush=True)
print("=" * 50, flush=True)
print("\nWaiting for jobs...", flush=True)

done_gen = done_ft = failed = 0

while True:
    try:
        r = requests.get(f"{API_URL}/api/v1/worker/next-finetune-job", headers=HEADERS, timeout=15)
        if r.status_code == 200:
            try:
                tt = handle_finetune_job(r.json()); done_ft += 1
                print(f"  Done {tt:.0f}s | gen={done_gen} ft={done_ft}", flush=True)
            except: failed += 1
            continue

        r = requests.get(f"{API_URL}/api/v1/worker/next-job", headers=HEADERS, timeout=15)
        if r.status_code == 200:
            job = r.json()
            try:
                gt = handle_generation_job(job); done_gen += 1
                print(f"  Done {gt:.1f}s | gen={done_gen} ft={done_ft}", flush=True)
            except Exception as e:
                failed += 1; print(f"  Error: {e}", flush=True)
                try: requests.post(f"{API_URL}/api/v1/worker/{job['job_id']}/result", headers=HEADERS,
                                  data={"status":"failed","error_message":str(e)[:500]}, timeout=10)
                except: pass
            continue

        time.sleep(POLL_INTERVAL)
    except KeyboardInterrupt:
        print(f"\nStop. gen={done_gen} ft={done_ft} err={failed}", flush=True); break
    except Exception as e:
        print(f"  Loop error: {e}", flush=True); time.sleep(5)

---
## Утилиты

In [ ]:
# Logs
try:
    with open("/content/ace_step_server.log") as f:
        lines = f.readlines()
        print(f"{len(lines)} lines")
        for l in lines[-30:]: print(l.rstrip())
except: print("No logs yet")

In [ ]:
# GPU status
import torch, psutil
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.memory_allocated()/1024**3:.1f}/{torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB")
print(f"RAM: {psutil.virtual_memory().used/1024**3:.1f}/{psutil.virtual_memory().total/1024**3:.1f} GB")
print(f"Disk: {psutil.disk_usage('/').used/1024**3:.1f}/{psutil.disk_usage('/').total/1024**3:.1f} GB")

In [ ]:
# Restart ACE-Step
import subprocess, torch
subprocess.run(["pkill", "-f", "acestep|ace_step"], capture_output=True)
torch.cuda.empty_cache()
print(f"Cleared. VRAM: {torch.cuda.memory_allocated()/1024**3:.2f} GB")
print("Run Step 4 again to restart.")